In [1]:
import os

os.environ["OPENAI_API_KEY"] = "sk-19fe6b9376f8473bab1defd0bde82559"
os.environ["MINERU_MODEL_SOURCE"] = "modelscope"

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
#     model="glm-5.1",
#     model="qwen3.5-plus",
    model="kimi-k2.5",
    temperature=0,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
def _bbox_area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def _looks_like_source_line(text):
    t = re.sub(r"\s+", "", text or "")
    return t.startswith(("来源：", "来源:", "资料来源：", "资料来源:", "数据来源：", "数据来源:"))

def _bbox_intersection(a, b):
    x0 = max(a[0], b[0])
    y0 = max(a[1], b[1])
    x1 = min(a[2], b[2])
    y1 = min(a[3], b[3])
    if x1 <= x0 or y1 <= y0:
        return 0.0
    return (x1 - x0) * (y1 - y0)

def _is_mostly_inside(a, b, thr=0.9):
    inter = _bbox_intersection(a, b)
    area = _bbox_area(a)
    return area > 0 and inter / area >= thr

def _filter_real_table_candidates(table_blocks, page_w):
    """
    去掉标题行、底部细线、被大表包含的小碎块。
    """
    items = []
    for blk in table_blocks:
        bbox = _collect_table_inner_bboxes(blk) or _safe_bbox(blk.get("bbox"))
        if not bbox:
            continue

        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]

        # 太窄/太矮，基本不是完整表
        if w < page_w * 0.45:
            continue
        if h < 40:
            continue

        items.append((blk, bbox, _bbox_area(bbox)))

    # 先保留大的
    items.sort(key=lambda x: x[2], reverse=True)

    kept = []
    kept_bboxes = []

    for blk, bbox, _ in items:
        if any(_is_mostly_inside(bbox, kb, thr=0.9) for kb in kept_bboxes):
            continue
        kept.append(blk)
        kept_bboxes.append(bbox)

    # 再按从上到下排序
    kept.sort(key=lambda blk: (_collect_table_inner_bboxes(blk) or _safe_bbox(blk.get("bbox")))[1])
    return kept

In [ ]:
def _iter_candidate_text_blocks(page_meta):
    """
    不只看 para_blocks，也把 blocks / preproc_blocks 里的文本块一起扫。
    并递归展开子 blocks。
    """
    roots = []
    for key in ("para_blocks", "blocks", "preproc_blocks"):
        roots.extend(page_meta.get(key, []) or [])

    seen = set()
    stack = list(roots)

    while stack:
        blk = stack.pop(0)

        for sub in blk.get("blocks", []) or []:
            stack.append(sub)

        bbox = _safe_bbox(blk.get("bbox"))
        text = _extract_block_text(blk)
        blk_type = blk.get("type", "")

        sig = (
            tuple(round(x, 1) for x in bbox) if bbox else None,
            blk_type,
            text[:100]
        )
        if sig in seen:
            continue
        seen.add(sig)

        yield blk
        
def _looks_like_table_title(text):
    text = re.sub(r"\s+", "", text or "")
    return bool(re.match(r"^(图表|表|图)\s*[0-9一二三四五六七八九十]+\s*[:：.．、]?", text))

def _iter_candidate_table_blocks(page_meta):
    """
    不只看 page_meta['tables']，
    也递归扫描 para_blocks / blocks / preproc_blocks 中的表格块。
    """
    roots = []
    for key in ("tables", "para_blocks", "blocks", "preproc_blocks"):
        roots.extend(page_meta.get(key, []) or [])

    seen = set()
    stack = list(roots)

    while stack:
        blk = stack.pop(0)

        subs = blk.get("blocks", []) or []
        for sub in subs:
            stack.append(sub)

        blk_type = str(blk.get("type", "") or "")

        has_table_child = any(
            str(sub.get("type", "") or "").startswith("table")
            for sub in subs
        )

        if not (blk_type.startswith("table") or has_table_child):
            continue

        bbox = _collect_table_inner_bboxes(blk) or _safe_bbox(blk.get("bbox"))
        if not bbox:
            continue

        sig = (
            tuple(round(x, 1) for x in bbox),
            blk_type,
        )
        if sig in seen:
            continue
        seen.add(sig)

        yield blk

In [ ]:
def parse_pdf_to_markdown(pdf_path, output_root, start_page_id=0, end_page_id=None):
    from mineru.cli.common import read_fn, prepare_env, convert_pdf_bytes_to_bytes
    from mineru.data.data_reader_writer import FileBasedDataWriter
    from mineru.backend.pipeline.pipeline_analyze import doc_analyze_streaming
    from mineru.backend.pipeline.pipeline_middle_json_mkcontent import union_make as pipeline_union_make
    from mineru.utils.enum_class import MakeMode
    import os
    import json

    pdf_bytes = read_fn(pdf_path)

    pdf_file_name = os.path.splitext(os.path.basename(pdf_path))[0]
    parse_method = "auto"

    local_image_dir, local_md_dir = prepare_env(output_root, pdf_file_name, parse_method)

    image_writer = FileBasedDataWriter(local_image_dir)
    md_writer = FileBasedDataWriter(local_md_dir)

    processed_bytes = convert_pdf_bytes_to_bytes(
        pdf_bytes,
        start_page_id=start_page_id,
        end_page_id=end_page_id
    )

    pdf_bytes_list = [processed_bytes]
    results = []

    def on_doc_ready(doc_index, model_list, middle_json, ocr_enable):
        results.append({
            "doc_index": doc_index,
            "model_list": model_list,
            "middle_json": middle_json,
            "ocr_enable": ocr_enable,
        })

    doc_analyze_streaming(
        pdf_bytes_list=pdf_bytes_list,
        image_writer_list=[image_writer],
        lang_list=["ch"],
        on_doc_ready=on_doc_ready,
        parse_method=parse_method,
        formula_enable=True,
        table_enable=True,
    )

    if not results:
        raise RuntimeError(f"MinerU 解析失败: {pdf_path}")

    middle_json = results[0]["middle_json"]
    pdf_info = middle_json["pdf_info"]
    image_dir = os.path.basename(local_image_dir)

    md_content_str = pipeline_union_make(
        pdf_info,
        MakeMode.MM_MD,
        image_dir
    )

    md_path = os.path.join(local_md_dir, f"{pdf_file_name}.md")
    md_writer.write_string(f"{pdf_file_name}.md", md_content_str)

    middle_json_path = os.path.join(local_md_dir, f"{pdf_file_name}.middle.json")
    with open(middle_json_path, "w", encoding="utf-8") as f:
        json.dump(middle_json, f, ensure_ascii=False, indent=2)

    return md_path, local_image_dir, middle_json_path

In [ ]:
def _ocr_title_band_above_table(page_img, table_bbox, sx, sy, band_height_pt=180):
    try:
        from rapidocr_onnxruntime import RapidOCR
    except Exception:
        return None

    engine = RapidOCR()

    x0 = 0
    x1 = page_img.width
    y1 = int(round(table_bbox[1] * sy))
    y0 = max(0, int(round((table_bbox[1] - band_height_pt) * sy)))

    band = page_img.crop((x0, y0, x1, y1))
    result, _ = engine(band)
    if not result:
        return None

    texts = []
    top_y = None

    for item in result:
        if len(item) < 2:
            continue
        box, txt = item[0], str(item[1])
        pure = re.sub(r"\s+", "", txt)
        if _looks_like_table_title(pure):
            texts.append(txt)
            ys = [p[1] for p in box]
            this_top = min(ys)
            top_y = this_top if top_y is None else min(top_y, this_top)

    if not texts:
        return None

    # OCR box 是相对 band 的坐标，要换回页面坐标
    top_page_y = (y0 + top_y) / sy
    return {
        "bbox": [0, top_page_y, table_bbox[2], table_bbox[1]],
        "text": " ".join(texts),
        "gap": table_bbox[1] - top_page_y,
        "is_caption_like": True,
    }

In [ ]:
def _find_caption_in_top_band(page, table_bbox, top_band=24):
    """
    查找落在表格顶边附近、但已经被并入 base_bbox 的标题行。
    例如：图表8：2025年国家医保目录新增7个中药产品
    """
    clip = fitz.Rect(
        0,
        max(0, table_bbox[1] - 4),
        page.rect.width,
        min(page.rect.height, table_bbox[1] + top_band),
    )

    words = page.get_text("words", clip=clip) or []
    if not words:
        return []

    words = sorted(words, key=lambda w: (round(w[1], 1), w[0]))

    lines = []
    current = []

    def flush():
        nonlocal current
        if not current:
            return
        xs0 = [w[0] for w in current]
        ys0 = [w[1] for w in current]
        xs1 = [w[2] for w in current]
        ys1 = [w[3] for w in current]
        text = "".join(str(w[4]) for w in current).strip()
        lines.append({
            "bbox": [min(xs0), min(ys0), max(xs1), max(ys1)],
            "text": text
        })
        current = []

    for w in words:
        if not current:
            current = [w]
            continue

        if abs(w[1] - current[-1][1]) <= 3:
            current.append(w)
        else:
            flush()
            current = [w]

    flush()

    hits = []
    for line in lines:
        text_no_space = re.sub(r"\s+", "", line["text"])
        if _looks_like_table_title(text_no_space):
            hits.append({
                "bbox": line["bbox"],
                "text": line["text"],
                "gap": 0,
                "is_caption_like": True,
            })

    hits.sort(key=lambda x: x["bbox"][1])
    return hits[:1]

In [ ]:
import os
import re
import json
import fitz
from PIL import Image

def _safe_bbox(bbox):
    if not bbox or len(bbox) != 4:
        return None
    x0, y0, x1, y1 = map(float, bbox)
    if x1 <= x0 or y1 <= y0:
        return None
    return [x0, y0, x1, y1]

def _union_bboxes(bboxes):
    bboxes = [b for b in bboxes if b is not None]
    if not bboxes:
        return None
    return [
        min(b[0] for b in bboxes),
        min(b[1] for b in bboxes),
        max(b[2] for b in bboxes),
        max(b[3] for b in bboxes),
    ]

def _extract_block_text(block):
    texts = []

    direct_txt = block.get("text") or block.get("content") or ""
    if direct_txt:
        texts.append(str(direct_txt))

    for line in block.get("lines", []) or []:
        for span in line.get("spans", []) or []:
            txt = span.get("content") or span.get("text") or ""
            if txt:
                texts.append(str(txt))

    for sub in block.get("blocks", []) or []:
        sub_txt = _extract_block_text(sub)
        if sub_txt:
            texts.append(sub_txt)

    text = " ".join(texts)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _x_overlap_ratio(a, b):
    overlap = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    denom = max(1.0, min(a[2] - a[0], b[2] - b[0]))
    return overlap / denom

def _collect_table_inner_bboxes(table_block):
    """
    优先把 table_body / table_caption / table_footnote 的 bbox 合起来；
    如果拿不到，再退回 table_block 自身 bbox。
    """
    inner = []
    for sub in table_block.get("blocks", []) or []:
        sub_type = sub.get("type")
        if sub_type in {"table_body", "table_caption", "table_footnote"}:
            b = _safe_bbox(sub.get("bbox"))
            if b:
                inner.append(b)

    if inner:
        return _union_bboxes(inner)

    return _safe_bbox(table_block.get("bbox"))

def _find_header_blocks_above_table(
    page_meta,
    table_bbox,
    max_gap=140,
    min_x_overlap=0.15,
    max_header_chars=200,
    max_header_lines=6,
):
    candidates = []

    for blk in _iter_candidate_text_blocks(page_meta):
        bbox = _safe_bbox(blk.get("bbox"))
        if not bbox:
            continue

        text = _extract_block_text(blk)
        if _looks_like_source_line(text):
            continue
        if not text:
            continue

        gap = table_bbox[1] - bbox[3]
        if gap < 0:
            continue

        overlap = _x_overlap_ratio(bbox, table_bbox)
        is_caption_like = _looks_like_table_title(text)
        h = bbox[3] - bbox[1]

        if is_caption_like:
            # 只允许很靠近表格上方的标题行
            if gap > 40:
                continue
            # 标题不应该是一整大块图/表
            if h > 35:
                continue
            if overlap < 0.05:
                continue
        else:
            if gap > max_gap:
                continue
            if overlap < min_x_overlap:
                continue
            if h > 30:
                continue
            if len(text) > max_header_chars:
                continue

        candidates.append({
            "bbox": bbox,
            "text": text,
            "gap": gap,
            "overlap": overlap,
            "is_caption_like": is_caption_like,
        })

    # 如果找到了“图表8：...”这种标题，只取最近的一条，直接返回
    caption_candidates = [x for x in candidates if x["is_caption_like"]]
    if caption_candidates:
        caption_candidates.sort(key=lambda x: x["gap"])
        return [caption_candidates[0]]

    # 没有 caption-like 再走普通文本逻辑
    candidates.sort(key=lambda x: (x["gap"], -x["bbox"][3]))

    selected = []
    current_top = table_bbox[1]

    for item in candidates:
        bbox = item["bbox"]
        gap = current_top - bbox[3]
        if 0 <= gap <= max_gap and _x_overlap_ratio(bbox, table_bbox) >= min_x_overlap:
            selected.append(item)
            current_top = min(current_top, bbox[1])

    selected.sort(key=lambda x: x["bbox"][1])
    return selected

def _find_header_blocks_above_table_from_pdf_text(
    page,
    table_bbox,
    max_gap=180,
    min_x_overlap=0.01,
):
    """
    如果 middle_json 里没找到标题，再直接从 PDF 文本层找。
    """
    candidates = []
    blocks = page.get_text("blocks") or []

    for blk in blocks:
        x0, y0, x1, y1, text = blk[:5]
        if _looks_like_source_line(text):
            continue
        
        bbox = _safe_bbox([x0, y0, x1, y1])
        if not bbox:
            continue

        text = re.sub(r"\s+", " ", str(text or "")).strip()
        
        if not text:
            continue

        gap = table_bbox[1] - bbox[3]
        if gap < 0 or gap > max_gap:
            continue

        overlap = _x_overlap_ratio(bbox, table_bbox)
        is_caption_like = _looks_like_table_title(text)

        if is_caption_like:
            if overlap < 0.0:
                continue
        else:
            if overlap < min_x_overlap:
                continue
            if len(text) > 200:
                continue

        candidates.append({
            "bbox": bbox,
            "text": text,
            "gap": gap,
            "overlap": overlap,
            "is_caption_like": is_caption_like,
        })

    candidates.sort(
        key=lambda x: (
            0 if x["is_caption_like"] else 1,
            x["gap"],
            -x["bbox"][3]
        )
    )

    selected = []
    current_top = table_bbox[1]

    for item in candidates:
        bbox = item["bbox"]
        gap = current_top - bbox[3]

        if item["is_caption_like"]:
            if 0 <= gap <= max_gap:
                selected.append(item)
                current_top = min(current_top, bbox[1])
        else:
            if 0 <= gap <= max_gap and _x_overlap_ratio(bbox, table_bbox) >= min_x_overlap:
                selected.append(item)
                current_top = min(current_top, bbox[1])

    selected.sort(key=lambda x: x["bbox"][1])
    return selected

def _find_header_line_above_table_from_pdf_words(
    page,
    table_bbox,
    max_gap=220,
):
    """
    从 PDF 的 words 层重组“表格上方的行”，查找像 '图表8：...' 的标题。
    比 blocks 更稳。
    """
    clip = fitz.Rect(
        0,
        max(0, table_bbox[1] - max_gap),
        page.rect.width,
        max(0, table_bbox[1] - 1)
    )

    words = page.get_text("words", clip=clip) or []
    # words: x0, y0, x1, y1, text, block_no, line_no, word_no
    if not words:
        return []

    # 先按 y、x 排序
    words = sorted(words, key=lambda w: (round(w[1], 1), w[0]))

    # 按“同一行”聚合
    lines = []
    current = []

    def flush():
        nonlocal current
        if not current:
            return
        xs0 = [w[0] for w in current]
        ys0 = [w[1] for w in current]
        xs1 = [w[2] for w in current]
        ys1 = [w[3] for w in current]
        text = "".join(str(w[4]) for w in current).strip()
        lines.append({
            "bbox": [min(xs0), min(ys0), max(xs1), max(ys1)],
            "text": text
        })
        current = []

    for w in words:
        if not current:
            current = [w]
            continue

        prev = current[-1]
        # y 接近就视为同一行
        if abs(w[1] - prev[1]) <= 3:
            current.append(w)
        else:
            flush()
            current = [w]

    flush()

    # 只保留像“图表8：...”的行
    candidates = []
    for line in lines:
        text = re.sub(r"\s+", "", line["text"])
        if _looks_like_table_title(text):
            gap = table_bbox[1] - line["bbox"][3]
            if 0 <= gap <= max_gap:
                candidates.append({
                    "bbox": line["bbox"],
                    "text": line["text"],
                    "gap": gap,
                    "is_caption_like": True,
                })

    candidates.sort(key=lambda x: x["gap"])
    return candidates[:1]

def crop_tables_with_outer_headers(
    pdf_path,
    middle_json_path,
    out_dir,
    dpi=220,
    max_gap=80,
    min_x_overlap=0.45,
    pad_left=8,
    pad_right=8,
    pad_top=80,
    pad_bottom=8,
):
    """
    依据 middle.json 的 tables + para_blocks，
    把“表格外的表头”一并裁进图片。
    """
    os.makedirs(out_dir, exist_ok=True)

    with open(middle_json_path, "r", encoding="utf-8") as f:
        middle = json.load(f)

    pdf_info = middle.get("pdf_info", []) or []
    doc = fitz.open(pdf_path)

    results = []
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    for page_idx, page_meta in enumerate(pdf_info):
        page = doc.load_page(page_idx)

        page_w, page_h = page_meta.get("page_size", [page.rect.width, page.rect.height])
        page_w, page_h = float(page_w), float(page_h)

        raw_tables = list(_iter_candidate_table_blocks(page_meta))
        print(f"[DEBUG] page {page_idx+1}: raw table candidates = {len(raw_tables)}", flush=True)

        tables = _filter_real_table_candidates(raw_tables, page_w)
        print(f"[DEBUG] page {page_idx+1}: real table candidates = {len(tables)}", flush=True)

        if not tables:
            continue

        pix = page.get_pixmap(matrix=matrix, alpha=False)
        page_img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        sx = page_img.width / float(page_w)
        sy = page_img.height / float(page_h)

        for t_idx, table_block in enumerate(tables):
            base_bbox = _collect_table_inner_bboxes(table_block) or _safe_bbox(table_block.get("bbox"))
            if not base_bbox:
                print(f"[DEBUG] page {page_idx+1} table {t_idx+1}: base_bbox is None, type={table_block.get('type')}", flush=True)
                continue

            # 先查“已经贴在表格顶边里”的标题
            header_items = _find_caption_in_top_band(
                page=page,
                table_bbox=base_bbox,
                top_band=24,
            )

            # 再查“表格上方”的标题行
            if not header_items:
                header_items = _find_header_line_above_table_from_pdf_words(
                    page=page,
                    table_bbox=base_bbox,
                    max_gap=max_gap + 80,
                )

            if not header_items:
                header_items = _find_header_blocks_above_table(
                    page_meta=page_meta,
                    table_bbox=base_bbox,
                    max_gap=max_gap,
                    min_x_overlap=min_x_overlap,
                )

            if not header_items:
                header_items = _find_header_blocks_above_table_from_pdf_text(
                    page=page,
                    table_bbox=base_bbox,
                    max_gap=max_gap + 40,
                    min_x_overlap=0.01,
                )

            if not header_items:
                ocr_hit = _ocr_title_band_above_table(
                    page_img, base_bbox, sx, sy, band_height_pt=max_gap + 80
                )
                if ocr_hit:
                    header_items = [ocr_hit]

            all_boxes = [base_bbox] + [x["bbox"] for x in header_items]
            crop_bbox = _union_bboxes(all_boxes)
            if not crop_bbox:
                continue

            crop_bbox = [
                max(0, crop_bbox[0] - pad_left),
                max(0, crop_bbox[1] - pad_top),
                min(page_w, crop_bbox[2] + pad_right),
                min(page_h, crop_bbox[3] + pad_bottom),
            ]

            px_box = (
                int(round(crop_bbox[0] * sx)),
                int(round(crop_bbox[1] * sy)),
                int(round(crop_bbox[2] * sx)),
                int(round(crop_bbox[3] * sy)),
            )

            crop_img = page_img.crop(px_box)

            out_name = f"page_{page_idx+1:03d}_table_{t_idx+1:02d}.png"
            out_path = os.path.join(out_dir, out_name)
            crop_img.save(out_path)

            print(f"\n[PAGE {page_idx+1} TABLE {t_idx+1}]", flush=True)
            print("base_bbox =", base_bbox, flush=True)

            if header_items:
                print("[HEADER HIT]", flush=True)
                for h in header_items:
                    print("  text =", h["text"], flush=True)
                    print("  bbox =", h["bbox"], flush=True)
            else:
                print("[HEADER MISS]", flush=True)

            results.append({
                "page_idx": page_idx,
                "table_idx": t_idx,
                "image_path": out_path,
                "table_bbox": base_bbox,
                "crop_bbox": crop_bbox,
                "header_text": " ".join(x["text"] for x in header_items).strip(),
            })

    doc.close()

    meta_path = os.path.join(out_dir, "table_crop_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return results

In [ ]:
md_root = r"md_reports"
pdf_folder = r"全部数据\正式数据\附件5：研报数据"

md_records = []

for report_type in os.listdir(pdf_folder):
    report_dir = os.path.join(pdf_folder, report_type)
    if not os.path.isdir(report_dir):
        continue

    for pdf_name in os.listdir(report_dir):
        if not pdf_name.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(report_dir, pdf_name)

        try:
            md_path, image_dir, middle_json_path = parse_pdf_to_markdown(pdf_path, md_root)
            
            table_crop_dir = os.path.join(os.path.dirname(md_path), "table_crops")
            
            table_crop_meta = crop_tables_with_outer_headers(
                pdf_path=pdf_path,
                middle_json_path=middle_json_path,
                out_dir=table_crop_dir,
                dpi=220,
                max_gap=140,
                min_x_overlap=0.15,
                pad_top=20,
                pad_left=8,
                pad_right=8,
                pad_bottom=8,
            )
            
            md_records.append({
                "report_type": report_type,
                "pdf_name": pdf_name,
                "pdf_path": pdf_path,
                "md_path": md_path,
                "image_dir": image_dir,
                "middle_json_path": middle_json_path,
                "table_crop_dir": table_crop_dir,
                "table_crop_meta_path": os.path.join(table_crop_dir, "table_crop_meta.json"),
            })
            print(f"[OK] {pdf_path}")
        except Exception as e:
            print(f"[FAIL] {pdf_path} -> {e}")

In [ ]:
import re
import pdfplumber
from langchain_core.documents import Document

def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

In [ ]:
import os
from mineru.cli.common import prepare_env

md_root = r"md_reports"
pdf_folder = r"全部数据\正式数据\附件5：研报数据"

def rebuild_md_records(pdf_folder, md_root, parse_method="auto"):
    md_records = []

    for report_type in os.listdir(pdf_folder):
        report_dir = os.path.join(pdf_folder, report_type)
        if not os.path.isdir(report_dir):
            continue

        for pdf_name in os.listdir(report_dir):
            if not pdf_name.lower().endswith(".pdf"):
                continue

            pdf_path = os.path.join(report_dir, pdf_name)
            pdf_file_name = os.path.splitext(pdf_name)[0]

            local_image_dir, local_md_dir = prepare_env(md_root, pdf_file_name, parse_method)
            md_path = os.path.join(local_md_dir, f"{pdf_file_name}.md")
            middle_json_path = os.path.join(local_md_dir, f"{pdf_file_name}.middle.json")

            table_crop_dir = os.path.join(local_md_dir, "table_crops")
            table_crop_meta_path = os.path.join(table_crop_dir, "table_crop_meta.json")

            if os.path.exists(md_path):
                md_records.append({
                    "report_type": report_type,
                    "pdf_name": pdf_name,
                    "pdf_path": pdf_path,
                    "md_path": md_path,
                    "image_dir": local_image_dir,
                    "middle_json_path": middle_json_path if os.path.exists(middle_json_path) else "",
                    "table_crop_dir": table_crop_dir if os.path.exists(table_crop_dir) else "",
                    "table_crop_meta_path": table_crop_meta_path if os.path.exists(table_crop_meta_path) else "",
                })
            else:
                print(f"[MISS] 没找到 md 文件: {md_path}")

    return md_records

md_records = rebuild_md_records(pdf_folder, md_root)
print(f"恢复完成，共 {len(md_records)} 条")
print(md_records[:2])

In [ ]:
import os
import re
from langchain_core.documents import Document

md_root = r"md_reports"
pdf_folder = r"全部数据\正式数据\附件5：研报数据"

IMAGE_PATTERN = re.compile(r'!\[[^\]]*\]\(([^)]+)\)')

def normalize_for_retrieval(text: str) -> str:
    text = str(text or "")
    text = text.lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[，。；：、“”‘’（）()【】\[\]{}<>《》,.;:!?！？\-—_]", "", text)
    return text

def build_image_proxy_docs(md_text, rec, context_window=400, use_ocr=True):
    """
    从 markdown 中找到图片引用，
    为每张图片构造一条可检索 Document。
    适合普通插图 / 图表 / 流程图等。
    """
    image_docs = []
    matches = list(IMAGE_PATTERN.finditer(md_text))

    for idx, m in enumerate(matches):
        rel_path = m.group(1).strip()
        img_name = os.path.basename(rel_path)

        candidates = [
            os.path.join(rec["image_dir"], img_name),
            os.path.normpath(os.path.join(os.path.dirname(rec["md_path"]), rel_path)),
            os.path.normpath(rel_path),
        ]
        image_path = next((p for p in candidates if os.path.exists(p)), None)
        if not image_path:
            continue

        left = max(0, m.start() - context_window)
        right = min(len(md_text), m.end() + context_window)
        surrounding_text = clean_text(md_text[left:right].replace(m.group(0), " "))

        ocr_text = run_ocr(image_path) if use_ocr else ""

        surrounding_text_norm = normalize_for_retrieval(surrounding_text)
        ocr_text_norm = normalize_for_retrieval(ocr_text)

        page_content = clean_text(f"""
        [图片检索文档]
        文件名：{rec.get('pdf_name', '')}
        报告类型：{rec.get('report_type', '')}
        图片名：{img_name}
        图片上下文：{surrounding_text}
        图片上下文归一化：{surrounding_text_norm}
        图片OCR：{ocr_text}
        图片OCR归一化：{ocr_text_norm}
        """)

        image_docs.append(
            Document(
                page_content=page_content,
                metadata={
                    "report_type": rec.get("report_type", ""),
                    "file_name": rec.get("pdf_name", ""),
                    "file_path": rec.get("pdf_path", ""),
                    "md_path": rec.get("md_path", ""),
                    "middle_json_path": rec.get("middle_json_path", ""),
                    "image_dir": rec.get("image_dir", ""),
                    "source": rec.get("md_path", ""),
                    "image_path": image_path,
                    "image_name": img_name,
                    "source_type": "image_proxy",
                    "doc_type": "research_report_image",
                    "chunk_id": f"{os.path.splitext(rec['pdf_name'])[0]}_img_{idx}",
                }
            )
        )

    return image_docs
# import os
# import re
# from pathlib import Path
# from langchain_core.documents import Document

# IMAGE_PATTERN = re.compile(r'!\[[^\]]*\]\(([^)]+)\)')

# _ocr_engine = None

# def read_text_file(path):
#     return Path(path).read_text(encoding="utf-8", errors="ignore")

# # def run_ocr(image_path):
# #     """
# #     先用 OCR 把图片中的文字提出来。
# #     没装 rapidocr_onnxruntime 也没关系，会自动返回空字符串。
# #     """
# #     global _ocr_engine
# #     try:
# #         from rapidocr_onnxruntime import RapidOCR
# #         if _ocr_engine is None:
# #             _ocr_engine = RapidOCR()

# #         result, _ = _ocr_engine(image_path)
# #         if not result:
# #             return ""

# #         texts = []
# #         for item in result:
# #             if isinstance(item, (list, tuple)) and len(item) >= 2:
# #                 texts.append(str(item[1]))
# #         return clean_text("\n".join(texts))
# #     except Exception:
# #         return ""
def run_ocr(image_path, verbose=False):
    """
    对图片做 OCR。
    优先把图片读成 ndarray 再送进 OCR，避免中文路径/相对路径问题。
    """
    global _ocr_engine

    try:
        from rapidocr_onnxruntime import RapidOCR
    except Exception as e:
        if verbose:
            print(f"[OCR IMPORT FAIL] {e}")
        return ""

    try:
        if _ocr_engine is None:
            _ocr_engine = RapidOCR()

        if not os.path.exists(image_path):
            if verbose:
                print(f"[OCR PATH NOT FOUND] {image_path}")
            return ""

        img = Image.open(image_path).convert("RGB")
        img_np = np.array(img)

        result, _ = _ocr_engine(img_np)
        if not result:
            if verbose:
                print(f"[OCR EMPTY RESULT] {image_path}")
            return ""

        texts = []
        for item in result:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                texts.append(str(item[1]))

        text = clean_text("\n".join(texts))
        if verbose:
            print(f"[OCR OK] {image_path}")
            print(text[:1000])

        return text

    except Exception as e:
        if verbose:
            print(f"[OCR RUN FAIL] {image_path} -> {e}")
        return ""


# # def build_image_proxy_docs(md_text, rec, context_window=400):
# #     """
# #     从 markdown 中找到图片引用，
# #     为每张图片构造一条可检索 Document。
# #     """
# #     image_docs = []
# #     matches = list(IMAGE_PATTERN.finditer(md_text))

# #     for idx, m in enumerate(matches):
# #         rel_path = m.group(1).strip()
# #         img_name = os.path.basename(rel_path)

# #         candidates = [
# #             os.path.join(rec["image_dir"], img_name),
# #             os.path.normpath(os.path.join(os.path.dirname(rec["md_path"]), rel_path)),
# #             os.path.normpath(rel_path),
# #         ]
# #         image_path = next((p for p in candidates if os.path.exists(p)), None)
# #         if not image_path:
# #             continue

# #         left = max(0, m.start() - context_window)
# #         right = min(len(md_text), m.end() + context_window)
# #         surrounding_text = clean_text(md_text[left:right].replace(m.group(0), " "))

# #         ocr_text = run_ocr(image_path)

# # #         page_content = clean_text(f"""
# # #         [图片检索文档]
# # #         文件名：{rec['pdf_name']}
# # #         图片名：{img_name}
# # #         图片上下文：{surrounding_text}
# # #         图片OCR：{ocr_text}
# # #         """)
# #         page_content = clean_text(f"""
# #             [表格图片检索文档]
# #             文件名：{rec.get('pdf_name', '')}
# #             报告类型：{rec.get('report_type', '')}
# #             页码：第{page_idx}页
# #             表格编号：{table_idx}
# #             表头：{header_text}
# #             表头归一化：{header_text_norm}
# #             表格OCR：{ocr_text}
# #             表格OCR归一化：{ocr_text_norm}
# #             """)

# #         image_docs.append(
# #             Document(
# #                 page_content=page_content,
# #                 metadata={
# #                     "report_type": rec["report_type"],
# #                     "file_name": rec["pdf_name"],
# #                     "file_path": rec["pdf_path"],
# #                     "md_path": rec["md_path"],
# #                     "middle_json_path": rec.get("middle_json_path", ""),
# #                     "image_dir": rec["image_dir"],
# #                     "source": rec["md_path"],
# #                     "image_path": image_path,
# #                     "image_name": img_name,
# #                     "source_type": "image_proxy",
# #                     "doc_type": "research_report_image",
# #                     "chunk_id": f"{os.path.splitext(rec['pdf_name'])[0]}_img_{idx}",
# #                 }
# #             )
# #         )

# #     return image_docs

In [ ]:
import json
import os
from pathlib import Path
from langchain_core.documents import Document

def normalize_metadata(meta: dict) -> dict:
    meta = dict(meta or {})

    defaults = {
        "report_type": "",
        "file_name": "",
        "file_path": "",
        "md_path": "",
        "image_dir": "",
        "middle_json_path": "",
        "table_image_dir": "",
        "table_crop_meta_path": "",
        "source": "",
        "doc_type": "",
        "chunk_id": "",
        "section_h1": "",
        "section_h2": "",
        "h1": "",
        "h2": "",
        "h3": "",
        "source_type": "",
        "image_path": "",
        "page_idx": -1,
        "table_idx": -1,
    }

    int_fields = {"page_idx", "table_idx"}

    for k, default_v in defaults.items():
        if k not in meta or meta[k] is None:
            meta[k] = default_v
        else:
            if k in int_fields:
                try:
                    meta[k] = int(meta[k])
                except:
                    meta[k] = -1
            else:
                if not isinstance(meta[k], str):
                    meta[k] = str(meta[k])

    return meta

# def build_table_crop_docs(rec, use_ocr=True):
#     """
#     把 table_crops 目录下的裁剪表格图，转成可检索 Document。
#     """
#     docs = []

#     meta_path = rec.get("table_crop_meta_path", "")
#     if not meta_path or not os.path.exists(meta_path):
#         return docs

#     try:
#         items = json.loads(Path(meta_path).read_text(encoding="utf-8"))
#     except Exception as e:
#         print(f"[WARN] 读取 table_crop_meta 失败: {meta_path} -> {e}")
#         return docs

#     for idx, item in enumerate(items):
#         image_path = item.get("image_path", "")
#         if not image_path or not os.path.exists(image_path):
#             continue

#         page_idx = int(item.get("page_idx", 0)) + 1
#         table_idx = int(item.get("table_idx", idx)) + 1
#         header_text = clean_text(item.get("header_text", ""))

#         ocr_text = run_ocr(image_path) if use_ocr else ""

#         page_content = clean_text(f"""
#         [表格图片检索文档]
#         文件名：{rec.get('pdf_name', '')}
#         报告类型：{rec.get('report_type', '')}
#         页码：第{page_idx}页
#         表格编号：{table_idx}
#         表头：{header_text}
#         表格OCR：{ocr_text}
#         """)

#         docs.append(
#             Document(
#                 page_content=page_content,
#                 metadata={
#                     "report_type": rec.get("report_type", ""),
#                     "file_name": rec.get("pdf_name", ""),
#                     "file_path": rec.get("pdf_path", ""),
#                     "md_path": rec.get("md_path", ""),
#                     "source": rec.get("pdf_path", ""),
#                     "image_path": image_path,
#                     "table_crop_meta_path": meta_path,
#                     "page_idx": page_idx,
#                     "table_idx": table_idx,
#                     "source_type": "table_crop",
#                     "doc_type": "table_crop_image",
#                 }
#             )
#         )

#     return docs
def build_table_crop_docs(rec, use_ocr=True):
    docs = []

    meta_path = rec.get("table_crop_meta_path", "")
    if not meta_path or not os.path.exists(meta_path):
        return docs

    try:
        items = json.loads(Path(meta_path).read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[WARN] 读取 table_crop_meta 失败: {meta_path} -> {e}")
        return docs

    for idx, item in enumerate(items):
        image_path = item.get("image_path", "")
        if not image_path or not os.path.exists(image_path):
            continue

        page_idx = int(item.get("page_idx", 0)) + 1
        table_idx = int(item.get("table_idx", idx)) + 1
        header_text = clean_text(item.get("header_text", ""))
        ocr_text = run_ocr(image_path) if use_ocr else ""

        header_text_norm = normalize_for_retrieval(header_text)
        ocr_text_norm = normalize_for_retrieval(ocr_text)

        page_content = clean_text(f"""
        [表格图片检索文档]
        文件名：{rec.get('pdf_name', '')}
        报告类型：{rec.get('report_type', '')}
        页码：第{page_idx}页
        表格编号：{table_idx}
        表头：{header_text}
        表头归一化：{header_text_norm}
        表格OCR：{ocr_text}
        表格OCR归一化：{ocr_text_norm}
        """)

        docs.append(
            Document(
                page_content=page_content,
                metadata={
                    "report_type": rec.get("report_type", ""),
                    "file_name": rec.get("pdf_name", ""),
                    "file_path": rec.get("pdf_path", ""),
                    "md_path": rec.get("md_path", ""),
                    "source": rec.get("pdf_path", ""),
                    "image_path": image_path,
                    "table_crop_meta_path": meta_path,
                    "page_idx": page_idx,
                    "table_idx": table_idx,
                    "source_type": "table_crop",
                    "doc_type": "table_crop_image",
                }
            )
        )

    return docs

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os

docs = []
image_docs = []
table_crop_docs = []

def read_text_file(path):
    return Path(path).read_text(encoding="utf-8", errors="ignore")

# for rec in md_records:
#     md_path = rec["md_path"]
#     report_type = rec["report_type"]
#     pdf_name = rec["pdf_name"]
#     pdf_path = rec["pdf_path"]
#     image_dir = rec["image_dir"]

#     md_text = read_text_file(md_path)
#     md_text = clean_text(md_text)

#     base_doc = Document(
#         page_content=md_text,
#         metadata={
#             "report_type": report_type,
#             "file_name": pdf_name,
#             "file_path": pdf_path,
#             "md_path": md_path,
#             "image_dir": image_dir,
#             "middle_json_path": rec.get("middle_json_path", ""),
#             "table_image_dir": rec.get("table_crop_dir", ""),
#             "table_crop_meta_path": rec.get("table_crop_meta_path", ""),
#             "source": md_path,
#             "doc_type": "research_report_md"
#         }
#     )
#     docs.append(base_doc)

#     image_docs.extend(build_image_proxy_docs(md_text, rec))
#     table_crop_docs.extend(build_table_crop_docs(rec))
for rec in md_records:
    md_text = read_text_file(rec["md_path"])
    md_text = clean_text(md_text)

    base_doc = Document(
        page_content=md_text,
        metadata={
            "report_type": rec["report_type"],
            "file_name": rec["pdf_name"],
            "file_path": rec["pdf_path"],
            "md_path": rec["md_path"],
            "image_dir": rec["image_dir"],
            "middle_json_path": rec.get("middle_json_path", ""),
            "table_image_dir": rec.get("table_crop_dir", ""),
            "table_crop_meta_path": rec.get("table_crop_meta_path", ""),
            "source": rec["md_path"],
            "doc_type": "research_report_md"
        }
    )
    docs.append(base_doc)

    # 先不加入普通图片代理文档
    # image_docs.extend(build_image_proxy_docs(md_text, rec))

    table_crop_docs.extend(build_table_crop_docs(rec))
    
print(f"table crop docs: {len(table_crop_docs)}")
for i, d in enumerate(table_crop_docs[:3]):
    print(f"\n--- table_crop_doc {i} ---")
    print(d.page_content[:2000])
    print(d.metadata)

print(docs[0].page_content[:1000])
print("-" * 100)
print(docs[0].metadata)

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

header_docs = []
for doc in docs:
    splits = header_splitter.split_text(doc.page_content)

    for s in splits:
        s.metadata = {
            **doc.metadata,
            **(s.metadata or {})
        }
        header_docs.append(s)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1200,
    chunk_overlap=200
)

doc_splits = text_splitter.split_documents(header_docs)

for i, doc in enumerate(doc_splits):
    meta = doc.metadata or {}
    file_stem = os.path.splitext(meta.get("file_name", "unknown.pdf"))[0]
    h1 = meta.get("h1", "")
    h2 = meta.get("h2", "")

    doc.metadata["chunk_id"] = f"{file_stem}_c{i}"
    doc.metadata["section_h1"] = h1
    doc.metadata["section_h2"] = h2
    doc.metadata["source_type"] = "markdown_chunk"
    doc.metadata["table_image_dir"] = meta.get("table_image_dir", "")
    doc.metadata["table_crop_meta_path"] = meta.get("table_crop_meta_path", "")

all_vector_docs = doc_splits + table_crop_docs

for d in all_vector_docs:
    d.metadata = normalize_metadata(d.metadata)

print(f"markdown chunks: {len(doc_splits)}")
print(f"image proxy docs: {len(image_docs)}")
print(f"table crop docs: {len(table_crop_docs)}")
print(f"all docs: {len(all_vector_docs)}")

In [ ]:
from langchain_milvus import Milvus
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.tools.retriever import create_retriever_tool

embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    show_progress=True,
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64,
    },
)

vectorstore = Milvus.from_documents(
    documents=all_vector_docs,
    embedding=embeddings,
    connection_args={"host": "localhost", "port": "19530"},
    collection_name="docs",
    drop_old=True,
)

In [ ]:
from langchain_milvus import Milvus
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.tools.retriever import create_retriever_tool

embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Milvus(
    embedding_function=embeddings,
    connection_args={"host": "localhost", "port": "19530"},
    collection_name="docs",
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20}
)

texts = retriever.invoke("国家医保目录新增的中药产品有哪些")

for text in texts:
    print(text.page_content)
    print("-" * 50)

In [ ]:
retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_pdf_file",
    "Search and return information from the internal knowledge base.",
) 

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

engine = create_engine("sqlite:///example.db")

###
sheets = pd.read_excel(
    r"final_result_final.xlsx",
    sheet_name=None,
    engine="openpyxl"
)
###

for sheet_name, df in sheets.items():
    table_name = sheet_name.strip().replace(" ", "_").replace("-", "_")
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)

db = SQLDatabase.from_uri("sqlite:///example.db")
print(db.get_usable_table_names())

In [ ]:
import os
import requests, pathlib
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI

from langchain.messages import AIMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
import json

In [ ]:
def merge_dict(a, b):
    a = a or {}
    b = b or {}
    out = dict(a)
    out.update(b)
    return out

def merge_list(a, b):
    return (a or []) + (b or [])

In [ ]:
from typing_extensions import TypedDict, Required
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from typing import List, Dict, Any, Literal, Annotated
from pydantic import BaseModel, Field
from pydantic import BaseModel, Field
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class SubTask(BaseModel):
    task_id: str
    type: Literal["sql", "rag", "synthesis", "clarify"]
    goal: str
    priority: int = Field(default=5, ge=1, le=10)
    depends_on: List[str] = Field(default_factory=list)
    status: Literal["pending", "ready", "running", "done", "failed"] = "pending"

class ParentState(MessagesState, total=False):
    query: Required[str]
        
    real_query: str

    reason: str
    missing_slots: List[str] = []
        
    task_plan: List[dict]
    
    task_results: Annotated[Dict[str, dict], merge_dict]
    execution_trace: Annotated[List[dict], merge_list]
        
    current_task_id: str
    schedule_status: Literal["all_done", "blocked", "has_task"]

    final_answer: str
    summary: str
        
    sql_messages: Annotated[list[AnyMessage], add_messages]
        
    image_path: List[str]
    references: List[dict]
    last_sql_query: str
        
    question_count: int
    skip_summary: bool
        
class PlanDecision(BaseModel):
    reason: str
    missing_slots: List[str] = Field(default_factory=list)
    tasks: List[SubTask] = Field(default_factory=list)

In [ ]:
def merge_query(state: ParentState):
    query = state.get("query") or state["messages"][-1].content
    summary = state.get("summary", "")
    
    if summary == "":
        return {
        "real_query": query,
        }
    
    prompt = f"""
    
你是一个对话查询改写助手。你的任务是根据“历史对话摘要 summary”和“用户当前输入 query”，判断用户这一轮真正想问的问题，并输出一个完整、明确、可独立理解的 query。

请严格遵循以下规则：

1. 如果当前 query 本身已经完整明确，不依赖历史上下文，就直接原样返回，不要过度改写。
2. 如果当前 query 中出现了代词、指代、省略、延续上文的表达（例如“它”“这个”“刚才那个”“继续”“再详细一点”“那数据库里的呢”），就结合 summary 补全成一个完整明确的问题。
3. 改写后的 query 必须保留用户原本意图，不能擅自增加新需求，也不能改变问题方向。
4. 优先补全以下信息：
   - 当前问的是哪个对象
   - 当前问的是哪个任务/主题
   - 是否是在上一轮结果基础上继续追问
   - 是否有输出格式、范围、限制条件
5. 如果 summary 与当前 query 无明显关系，则以当前 query 为准。
6. 输出必须简洁，只输出“最终 query”本身，不要输出解释，不要输出分析过程，不要加引号。

历史对话摘要 summary：
{summary if summary else "（无历史摘要）"}

用户当前输入 query：
{query}

请输出用户这一轮真正的完整 query：
    
    """
    
    real_query = llm.invoke([{"role": "user", "content": prompt}]).content.strip()
    
    return {
        "real_query": real_query,
    }

In [ ]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

get_schema_tool = next(tool for tool in tools if tool.name == "sql_db_schema")
get_schema_node = ToolNode([get_schema_tool], name="get_schema", messages_key="sql_messages")

run_query_tool = next(tool for tool in tools if tool.name == "sql_db_query")
run_query_node = ToolNode([run_query_tool], name="run_query", messages_key="sql_messages")

In [ ]:
def list_tables(state: ParentState):
    
    usable_tables = list(db.get_usable_table_names())
    
    query = state["real_query"]
    
    prompt = f"""
    
你需要为后续 SQL 查询选择最相关的表，并调用 sql_db_schema 获取 schema。

当前数据库可选表只有：
{usable_tables}

要求：
1. 只能从上面这些表里选
2. 必须选择 1~3 张表，不能返回空列表
3. 不要直接回答用户问题
4. 如果问利润、收入、净利润、每股收益，优先考虑 income
5. 如果问资产、负债，优先考虑 balance
6. 如果问现金流，优先考虑 cash

当前问题：{query}
"""
    llm_with_tools = llm.bind_tools([get_schema_tool], tool_choice="any")
    response = llm_with_tools.invoke([HumanMessage(content=prompt)])

    return {"sql_messages": [response]}

generate_query_system_prompt = """
你是一个专为与 SQL 数据库交互而设计的智能体。
给定一个用户输入问题，请生成一条语法正确的 {dialect} 查询语句并执行，
然后查看查询结果并返回答案。除非用户明确指定了希望获取的示例数量，否则始终将你的查询结果限制在最多 {top_k} 条。

你可以按相关列对结果进行排序，以返回数据库中最有趣的示例。切勿查询特定表的所有列，仅根据问题请求相关的列。

对于趋势/可视化类问题（例如：趋势、变化、走势、近几年、历年、可视化、绘图、折线图、柱状图）：
- 返回时间序列查询，而非单个标量结果
- 选择一个时间周期列和一个数值指标列
- 尽可能将它们分别别名为 x_axis 和 y_axis
- 按时间顺序升序排列结果
- 不要机械地应用默认的 top_k 限制；应保留绘图所需的所有时间段

切勿对数据库执行任何 DML 语句（如 INSERT、UPDATE、DELETE、DROP 等）。

如果上下文中已经给出了所需表的 schema，请不要再次调用 sql_db_schema。
此阶段只允许两种行为：
1. 生成并调用 sql_db_query；
2. 在已有查询结果时直接输出最终答案。

【最终回答格式要求】
1. 只输出纯文本，不要使用 Markdown。
2. 不要使用表格语法，如 |---|、| 列 |。
3. 不要使用 **粗体**、# 标题、- 列表。
4. 先给结论，再给必要说明。
5. 输出内容要适合直接写入 Excel 单元格。
6. 如果问题属于趋势/变化/走势/历年分析类，不要只给一句笼统结论，必须结合查询结果展开说明。
7. 对于趋势类问题，回答中尽量包含以下信息：
   - 总体趋势判断（上升、下降、波动、先升后降等）
   - 各时间段的关键数值变化
   - 相邻时期的增减变化
   - 全周期累计变化幅度
   - 如数据明显连续增长或连续下降，要明确指出
8. 如果查询结果是多期时间序列，回答不少于3句，避免只输出一句压缩总结。
9. 回答仍然使用纯文本，但可以分句清晰表达，不要过度简写。
""".format(
    dialect=db.dialect,
    top_k=5,
)

def generate_query(state: ParentState):
    query = state["real_query"]
    
    system_message = {
        "role": "system",
        "content": generate_query_system_prompt + f"""

            补充上下文：
            - 用户输入问题：{query}
    
            """
    }
    
    llm_with_tools = llm.bind_tools([run_query_tool])
    response = llm_with_tools.invoke([system_message] + state["sql_messages"])

    return {"sql_messages": [response]}


check_query_system_prompt = """
你是一位注重细节的 SQL 专家。

请仔细检查 {dialect} 查询中是否存在以下常见错误：
- 在涉及 NULL 值时使用 NOT IN
- 应使用 UNION ALL 时却使用了 UNION
- 使用 BETWEEN 处理开区间（排他性范围）
- 谓词中的数据类型不匹配
- 标识符引用未正确加引号
- 函数使用的参数数量不正确
- 未转换为正确的数据类型
- 连接（JOIN）时使用了错误的列

如果发现上述任何错误，请重写查询语句；如果没有错误，则直接复现原始查询。

完成此检查后，你将调用相应的工具来执行查询。
""".format(dialect=db.dialect)

def check_query(state: ParentState):
    system_message = {
        "role": "system",
        "content": check_query_system_prompt,
    }

    tool_call = state["sql_messages"][-1].tool_calls[0]
    tool_name = tool_call.get("name")

    if tool_name != "sql_db_query":
        raise ValueError(f"check_query 只应处理 sql_db_query，当前收到: {tool_call}")

    args = tool_call.get("args", {}) or {}
    sql_text = args if isinstance(args, str) else (
        args.get("query") or
        args.get("__arg1") or
        args.get("input") or
        args.get("sql")
    )

    if not sql_text:
        raise ValueError(f"sql_db_query 的参数结构异常: {tool_call}")

    user_message = {"role": "user", "content": sql_text}

    llm_with_tools = llm.bind_tools([run_query_tool], tool_choice="any")
    response = llm_with_tools.invoke([system_message, user_message])
    response.id = state["sql_messages"][-1].id

    return {"sql_messages": [response]}

In [ ]:
def extract_table_names_from_tool_call(tool_call):
    args = tool_call.get("args", {}) or {}
    table_names = args.get("table_names") or args.get("__arg1") or []

    if isinstance(table_names, str):
        table_names = table_names.replace("，", ",")
        table_names = [x.strip() for x in table_names.split(",") if x.strip()]

    if table_names is None:
        table_names = []

    return table_names


def should_continue(state: ParentState) -> Literal["check_query", "get_schema", END]:
    last_message = state["sql_messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None) or []

    if not tool_calls:
        return END

    tool_call = tool_calls[0]
    tool_name = tool_call.get("name")

    if tool_name == "sql_db_query":
        return "check_query"

    if tool_name == "sql_db_schema":
        table_names = extract_table_names_from_tool_call(tool_call)
        if table_names:
            return "get_schema"

        print(f"[warn] sql_db_schema 的 table_names 为空: {tool_call}")
        return END

    print(f"[warn] generate_query 返回了未预期的工具调用: {tool_calls}")
    return END

builder = StateGraph(ParentState)
builder.add_node(list_tables)
# builder.add_node(call_get_schema)
builder.add_node(get_schema_node, "get_schema")
builder.add_node(generate_query)
builder.add_node(check_query)
builder.add_node(run_query_node, "run_query")

builder.add_edge(START, "list_tables")
# builder.add_edge("list_tables", "call_get_schema")
# builder.add_edge("call_get_schema", "get_schema")
builder.add_edge("list_tables", "get_schema")
builder.add_edge("get_schema", "generate_query")
builder.add_conditional_edges(
    "generate_query",
    should_continue,
    {
        "check_query": "check_query",
        "get_schema": "get_schema",
        END: END,
    },
)
builder.add_edge("check_query", "run_query")
builder.add_edge("run_query", "generate_query")

sql_graph = builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(sql_graph.get_graph().draw_mermaid_png()))

In [ ]:
def format_messages_as_text(messages):
    lines = []

    for msg in messages:
        if isinstance(msg, dict):
            role = msg.get("role", "unknown")
            content = msg.get("content", "")
        else:
            msg_type = getattr(msg, "type", "unknown")
            content = getattr(msg, "content", "")

            if msg_type == "human":
                role = "user"
            elif msg_type == "ai":
                role = "assistant"
            else:
                role = msg_type

        lines.append(f"{role}: {content}")

    return "\n".join(lines)

def summarize(state: ParentState):
    if state.get("skip_summary", False):
        return {}
    
    messages = state.get("messages", [])
    latest_turn = messages[-2:] if len(messages) >= 2 else messages
    latest_turn_text = format_messages_as_text(latest_turn)
    old_summary = state.get("summary", "")
    
    summarize_prompt = f"""
    
        你是一个负责维护多轮对话记忆的模块。
        
        请基于【历史记忆】和【最新对话】，更新记忆。
        只保留对后续任务推进有价值的信息。
        
        记忆中只保留：
        - 用户当前目标
        - 已确认的关键信息
        - 用户偏好与限制
        - 尚未解决的问题
        - 下一步应做的事

        请遵守：
        - 不要复述原始对话
        - 不要保留寒暄、客套、重复内容
        - 若新旧信息冲突，以最新且明确确认的信息为准
        - 未确认内容单独标注，不要当作事实
        - 删除过时信息
        - 不要编造任何未出现的信息
        - 回答最多使用四句话，并保持简洁。

        【历史信息】
        {old_summary}

        【最新对话】
        {latest_turn_text}
        
    """
    
    summarize_history = llm.invoke([{"role": "user", "content": summarize_prompt}]).content.strip()
    
    print("-"*100)
    print("Summary")
    print(summarize_history)
    print("-"*100)
    
    return {
        "summary": summarize_history,
    }

In [ ]:
import json
import re
from pydantic import ValidationError

VALID_TASK_TYPES = {"sql", "rag", "clarify", "synthesis"}
VALID_STATUS = {"pending", "ready", "running", "done", "failed"}

def extract_json_object(text: str) -> str:
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("planner 输出中未找到合法 JSON 对象")

    return match.group(0)


def normalize_plan_data(data: dict) -> dict:
    data = dict(data or {})
    data.setdefault("reason", "")
    data.setdefault("missing_slots", [])
    data.setdefault("tasks", [])

    tasks = data.get("tasks", [])
    normalized_tasks = []
    seen_task_ids = []

    for i, task in enumerate(tasks, start=1):
        task = dict(task or {})

        task_id = str(task.get("task_id") or f"task_{i}")
        task_type = str(task.get("type") or "clarify")
        if task_type not in VALID_TASK_TYPES:
            task_type = "clarify"

        goal = str(task.get("goal") or "").strip()
        if not goal:
            goal = f"执行{task_type}任务"

        try:
            priority = int(task.get("priority", i))
        except Exception:
            priority = i
        priority = max(1, min(priority, 10))

        depends_on = task.get("depends_on", [])
        if not isinstance(depends_on, list):
            depends_on = []

        # 只保留已经出现过的 task_id，避免非法依赖
        depends_on = [d for d in depends_on if d in seen_task_ids]

        status = str(task.get("status") or "pending")
        if status not in VALID_STATUS:
            status = "pending"

        normalized_tasks.append({
            "task_id": task_id,
            "type": task_type,
            "goal": goal,
            "priority": priority,
            "depends_on": depends_on,
            "status": status,
        })
        seen_task_ids.append(task_id)

    # 自动补 synthesis：
    # 若存在多个执行任务(sql/rag)，且没有 clarify，且没生成 synthesis，则自动补一个
    exec_task_ids = [t["task_id"] for t in normalized_tasks if t["type"] in {"sql", "rag"}]
    has_clarify = any(t["type"] == "clarify" for t in normalized_tasks)
    has_synthesis = any(t["type"] == "synthesis" for t in normalized_tasks)

    if len(exec_task_ids) >= 2 and (not has_clarify) and (not has_synthesis):
        normalized_tasks.append({
            "task_id": f"task_{len(normalized_tasks) + 1}",
            "type": "synthesis",
            "goal": "整合前置任务结果，生成统一答复",
            "priority": min(10, len(normalized_tasks) + 1),
            "depends_on": exec_task_ids,
            "status": "pending",
        })

    data["tasks"] = normalized_tasks
    return data


def fallback_plan(question: str, err: Exception) -> PlanDecision:
    return PlanDecision(
        reason=f"planner 调用或解析失败：{err}",
        missing_slots=["period"],
        tasks=[
            SubTask(
                task_id="task_1",
                type="clarify",
                goal="询问用户想查询的时间范围",
                priority=1,
                depends_on=[],
                status="pending",
            )
        ],
    )

In [ ]:
def plan_node(state: ParentState):
    question = state["real_query"]


    route_prompt = f"""
你是上市公司财报智能问数系统的任务规划器（Planner）。

你的任务不是直接回答用户问题，而是：
1. 识别用户问题中的一个或多个核心意图；
2. 先在内部判断该问题整体更接近 sql / rag / hybrid / clarify；
3. 再将问题拆解为可执行 tasks；
4. 输出可直接进入调度器的 JSON。

你只能使用以下 task type：
- sql：适用于可以主要通过结构化财务数据库回答的问题，如数值查询、指标查询、同比/环比、趋势、排名、topN、筛选、排序、聚合统计、跨公司比较、可视化所需时间序列数据
- rag：适用于需要主要依赖非结构化文本知识库回答的问题，如原因分析、政策解读、研报观点、公告理解、行业判断、风险分析、管理层表述、市场预期
- clarify：适用于用户问题缺少关键条件，当前无法可靠执行
- synthesis：适用于同时存在多个执行任务时，对前置任务结果进行整合，形成统一答复

内部判定原则（非常重要）：
- 如果问题主要在问“多少、是否、排名、变化趋势、同比环比、topN、哪个最大/最小”，优先视为 sql
- 如果问题主要在问“为什么、原因、影响、怎么看、风险、政策、研报怎么说”，优先视为 rag
- 如果问题同时包含“查数据 + 做解释/归因/结合研报”，视为 hybrid
- 如果问题缺少关键查询条件，视为 clarify

请特别注意：
- 不要因为问题里出现“分析”两个字就一律归为 rag
- “趋势分析”“同比分析”“排名分析”“变化趋势”通常仍然属于 sql
- “原因分析”“政策影响分析”“研报观点分析”通常属于 rag
- “数据趋势 + 原因解释”属于 hybrid
- 若问题中有多个子任务，先判断每个子任务分别依赖结构化数据还是非结构化文本；只有同时存在至少一个 sql 子任务和至少一个 rag 子任务时，才应规划为混合多任务
- 对趋势/走势/变化/历年/可视化类问题，只要已经给出相对时间范围，就不要因为 period 再发 clarify

时间范围判定规则（非常重要）：
以下表达都视为已提供有效时间范围：
- 近几年
- 最近几年
- 近年来
- 近三年 / 近五年 / 过去三年 / 过去五年
- 历年 / 近年
- 最近几个报告期

例如：
- “华润三九近几年的每股收益变化趋势”
- “比亚迪近三年营收走势”
- “近年来净利润波动情况”
这些都应直接规划为 sql，不要再追问 period。

槽位缺失判断：
- company：公司名
- period：报告期/时间范围
- metric：指标名
- compare_target：比较对象
- explanation_target：待解释对象

clarify 触发原则：
- 缺少公司名，且无法从上下文确定
- 缺少指标名，且无法执行
- 完全缺少时间范围，且该问题明显依赖报告期
- 比较类问题缺少比较对象
- 原因解释类问题缺少待解释对象

任务规划规则：
1. 若问题包含多个核心意图，必须拆成多个 tasks
2. 若任务之间有前后依赖，必须写入 depends_on
3. priority 越小优先级越高
4. status 初始统一为 "pending"
5. 若存在多个执行任务(sql/rag)，通常最后要有 synthesis 任务
6. task_id 必须唯一，如 task_1, task_2, task_3
7. depends_on 只能引用前面已经出现的 task_id
8. 若整体属于 clarify，通常只输出一个 clarify task
9. 若整体本质是单一 sql 或单一 rag，就不要为了形式额外生成 synthesis
10. 只要后一步显式依赖前一步得到的对象集合、筛选结果、排名结果、统计结果，即使它们都属于 sql，也必须拆成多个 sql tasks
11. 对“topN/筛选结果 -> 查询这些对象的更多指标 -> 再比较/再排序/找最大最小”这类问题，默认拆为至少 3 个 tasks
12. 若问题中出现“这些企业/这些公司/其中/上述企业/上述结果”等代词回指，默认说明存在依赖链，应拆分
13. 不要把可以自然拆成多步链式执行的复杂 sql 问题压缩成一个笼统的 sql task

输出要求：
1. 只能输出 JSON
2. 不要输出 markdown 代码块
3. 不要输出额外解释
4. 最终只输出以下字段：reason、missing_slots、tasks

输出格式：
{{
  "reason": "为什么这样规划",
  "missing_slots": ["缺失槽位1", "缺失槽位2"],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "sql | rag | clarify | synthesis",
      "goal": "子任务目标",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例1：
用户问题：比亚迪2025年三季度营业收入是多少？
输出：
{{
  "reason": "这是明确的单指标单时期数值查询，主要依赖结构化财务数据库",
  "missing_slots": [],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "sql",
      "goal": "查询比亚迪2025年三季度营业收入",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例2：
用户问题：金花股份近几年的利润总额变化趋势是什么样的？
输出：
{{
  "reason": "这是时间序列趋势分析问题，且“近几年”已提供有效时间范围，主要依赖结构化财务数据",
  "missing_slots": [],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "sql",
      "goal": "查询金花股份近几年的利润总额时间序列数据，用于分析变化趋势",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例3：
用户问题：国家医保目录新增的中药产品有哪些？
输出：
{{
  "reason": "这是知识检索类问题，需要依赖政策文件或研报等非结构化文本知识库",
  "missing_slots": [],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "rag",
      "goal": "检索医保目录与相关资料中新增中药产品信息",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例4：
用户问题：华润三九近三年的主营业务收入情况做可视化绘图，主营业务收入上升的原因是什么？
输出：
{{
  "reason": "前半部分需要结构化财务数据做趋势分析和可视化，后半部分需要非结构化文本证据做原因解释，因此应拆分为混合多任务",
  "missing_slots": [],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "sql",
      "goal": "查询华润三九近三年主营业务收入时间序列数据，并用于可视化绘图",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }},
    {{
      "task_id": "task_2",
      "type": "rag",
      "goal": "检索华润三九主营业务收入上升原因的公告、年报或研报证据",
      "priority": 2,
      "depends_on": ["task_1"],
      "status": "pending"
    }},
    {{
      "task_id": "task_3",
      "type": "synthesis",
      "goal": "整合主营业务收入趋势结果、可视化结果与原因解释，形成统一答复",
      "priority": 3,
      "depends_on": ["task_1", "task_2"],
      "status": "pending"
    }}
  ]
}}

示例5：
用户问题：利润总额是多少？
输出：
{{
  "reason": "缺少公司和报告期，无法可靠执行查询",
  "missing_slots": ["company", "period"],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "clarify",
      "goal": "向用户确认公司名称和报告期，以继续查询利润总额",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例6：
用户问题：华为每股收益是多少？
输出：
{{
  "reason": "缺少报告期，无法直接执行查询",
  "missing_slots": ["period"],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "clarify",
      "goal": "向用户确认报告期，以继续查询每股收益",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

示例7：
用户问题：华润三九近几年的每股收益的变化趋势
输出：
{{
  "reason": "这是趋势分析问题，且“近几年”已提供有效时间范围，可直接通过结构化财务数据库查询",
  "missing_slots": [],
  "tasks": [
    {{
      "task_id": "task_1",
      "type": "sql",
      "goal": "查询华润三九近几年的每股收益时间序列数据，用于分析变化趋势",
      "priority": 1,
      "depends_on": [],
      "status": "pending"
    }}
  ]
}}

现在请对下面这个用户问题生成计划：
用户问题：{question}
""".strip()

    print(">>> enter plan_node")
    print(">>> before invoke")

    try:
        response = llm.invoke(route_prompt)
        print(">>> after invoke")

        raw = response.content.strip()
        json_str = extract_json_object(raw)
        data = json.loads(json_str)
        data = normalize_plan_data(data)
        decision = PlanDecision.model_validate(data)

    except Exception as e:
        print(f">>> first parse failed: {e}")

        # 只在首次失败时再补一次“修复 JSON”调用，平时不增加额外开销
        try:
            repair_prompt = f"""
请把下面内容修复为一个合法 JSON，且必须符合以下要求：
1. 顶层字段只能有 reason、missing_slots、tasks
2. tasks 中每项必须包含 task_id、type、goal、priority、depends_on、status
3. type 只能是 sql、rag、clarify、synthesis
4. status 统一写 pending
5. 只能输出 JSON，不要解释

原始内容：
{response.content}
""".strip()

            repaired = llm.invoke(repair_prompt)
            repaired_json = extract_json_object(repaired.content)
            data = json.loads(repaired_json)
            data = normalize_plan_data(data)
            decision = PlanDecision.model_validate(data)

        except Exception as e2:
            print(f">>> repair failed: {e2}")
            decision = fallback_plan(question, e2)

    return {
        "real_query": question,
        "reason": decision.reason,
        "missing_slots": decision.missing_slots,
        "task_plan": [task.model_dump() for task in decision.tasks],
        "current_task_id": "",
        "schedule_status": "blocked",
        "task_results": {},
        "execution_trace": [],
    }

In [ ]:
def get_current_task(state: ParentState):
    task_id = state["current_task_id"]
    for task in state.get("task_plan", []):
        if task_id == task["task_id"]:
            return task
    raise ValueError(f"找不到 current_task_id={task_id} 对应的任务")

In [ ]:
def rag_node(state: ParentState):
    task = get_current_task(state)
    goal = task["goal"]

    docs = retriever.invoke(goal)

    docs_text_list = []
    references = []
    image_hits = []

    for doc in docs:
        metadata = getattr(doc, "metadata", {}) or {}
        source_type = metadata.get("source_type", "")

        if source_type == "image_proxy":
            docs_text_list.append(f"[图片检索结果]\n{doc.page_content}")
        elif source_type == "table_crop":
            docs_text_list.append(f"[表格图片检索结果]\n{doc.page_content}")
        else:
            docs_text_list.append(doc.page_content)

        ref_item = {
            "text": doc.page_content[:500],
            "source_type": source_type,
        }

        paper_path = metadata.get("source", "")
        paper_image = metadata.get("image_path", "")

        if paper_path:
            ref_item["paper_path"] = paper_path

        if paper_image:
            ref_item["paper_image"] = paper_image
            image_hits.append(paper_image)

        references.append(ref_item)

    docs_text = "\n\n".join(docs_text_list)

    return {
        "task_results": {
            task["task_id"]: {
                "type": "rag",
                "goal": goal,
                "docs_text": docs_text,
                "references": references,
                "image_path": sorted(set(image_hits)),
                "status": "done",
            }
        },
        "execution_trace": [{
            "task_id": task["task_id"],
            "type": "rag",
            "goal": goal,
            "status": "done"
        }]
    }

# def rag_node(state: ParentState):
#     task = get_current_task(state)
#     goal = task["goal"]

#     docs = retriever.invoke(goal)
#     docs_text = "\n\n".join([doc.page_content for doc in docs])

#     references = []
#     image_hits = []
    
#     for doc in docs:
#         metadata = getattr(doc, "metadata", {}) or {}
#         source_type = metadata.get("source_type", "")
        
#         if source_type == "image_proxy":
#             docs_text_list.append(f"[图片检索结果]\n{doc.page_content}")
#         else:
#             docs_text_list.append(doc.page_content)

#         ref_item = {
#             "text": doc.page_content[:500],
#             "source_type": source_type,
#         }

#         paper_path = metadata.get("source", "")
# #         page_no = metadata.get("page", "")
#         paper_image = metadata.get("image_path", "")
    
#         if paper_path:
#             ref_item["paper_path"] = paper_path

#         if image_path:
#             ref_item["paper_image"] = image_path
#             image_hits.append(image_path)

#         references.append(ref_item)
        
#     docs_text = "\n\n".join(docs_text_list)

#     return {
#         "task_results": {
#             task["task_id"]: {
#                 "type": "rag",
#                 "goal": goal,
#                 "docs_text": docs_text,
#                 "references": references,
#                 "image_path": sorted(set(image_hits)),
#                 "status": "done",
#             }
#         },
#         "execution_trace": [{
#             "task_id": task["task_id"],
#             "type": "rag",
#             "goal": goal,
#             "status": "done"
#         }]
#     }

In [ ]:
def clarify(state: ParentState):
    task = get_current_task(state)
    question = state["real_query"]
    missing_slots = state.get("missing_slots", [])

    clarify_prompt = f"""
        你是一个上市公司财报智能问数助手。

        用户原始问题：
        {question}

        当前还缺少这些关键信息：
        {missing_slots}

        你的任务是：
        基于用户原始问题和缺失槽位，生成一句自然、简洁、礼貌的澄清问题，引导用户一次性补全所缺信息，以便继续查询。

        要求：
        1. 只输出一句面向用户的澄清问题，不要输出解释，不要输出 JSON，不要输出多余内容。
        2. 如果缺少多个槽位，尽量合并成一句话一起问，不要拆成多句。
        3. 语气自然，适合中文对话场景。
        4. 问法要贴合财报问数场景。
        
        示例1
        原始问题：利润总额是多少？
        缺失槽位：["company", "period"]
        输出：请问你想查询哪家公司，以及哪个报告期的利润总额？

        示例2
        原始问题：同比最高的是哪家？
        缺失槽位：["metric", "period", "ranking_scope"]
        输出：请问你想查询哪个指标、哪个报告期，以及希望在哪个范围内比较同比？

        示例3
        原始问题：原因是什么？
        缺失槽位：["explanation_target"]
        输出：请问你具体想了解哪个指标或现象变化的原因？
        """.strip()

    response = llm.invoke([{"role": "user", "content": clarify_prompt}])

    return {
        "messages": [AIMessage(content=response.content)],
        "final_answer": response.content,
        "missing_slots": missing_slots,
        "task_plan": update_task_status(state["task_plan"], task["task_id"], "done"),
        "task_results": {
            task["task_id"]: {
                "type": "clarify",
                "goal": task["goal"],
                "answer": response.content,
                "status": "done",
            }
        },
        "execution_trace": [{
            "task_id": task["task_id"],
            "type": "clarify",
            "goal": task["goal"],
            "status": "done",
        }]
    }

In [ ]:
import re

def clean_answer_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # 统一换行
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 去掉代码块
    text = re.sub(r"```[\s\S]*?```", "", text)

    # 去掉 markdown 强调/标题
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.M)

    # 去掉无序列表符号
    text = re.sub(r"^\s*[-*+]\s*", "", text, flags=re.M)

    # 去掉多余空白，把换行压成空格
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()

In [ ]:
def merge_generate(state: ParentState):
    question = state["real_query"]
    task_results = state.get("task_results", {})
    current_task_id = state.get("current_task_id", "")

    sql_parts = []
    rag_parts = []
    images = []
    all_references = []

    for task_id, result in task_results.items():
        if result.get("type") == "sql":
            sql_parts.append(
                f"[{task_id}] 目标: {result.get('goal', '')}\n"
                f"SQL: {result.get('query', '')}\n"
                f"结果: {result.get('answer', '')}"
            )
            images.extend(result.get("image_path", []))

#         elif result.get("type") == "rag":
#             rag_parts.append(
#                 f"[{task_id}] 目标: {result.get('goal', '')}\n"
#                 f"检索内容: {result.get('docs_text', '')}"
#             )
#             all_references.extend(result.get("references", []))
        elif result.get("type") == "rag":
            rag_parts.append(
                f"[{task_id}] 目标: {result.get('goal', '')}\n"
                f"检索内容: {result.get('docs_text', '')}"
            )
            all_references.extend(result.get("references", []))
            images.extend(result.get("image_path", []))

    prompt = f"""
    你是一个财报智能助手。请根据已经完成的子任务结果，形成统一、连贯、去重后的最终回答。

    用户问题：
    {question}

    SQL子任务结果：
    {chr(10).join(sql_parts) if sql_parts else "无"}

    RAG子任务结果：
    {chr(10).join(rag_parts) if rag_parts else "无"}

    要求：
    1. 若有 SQL 结果，优先作为事实依据
    2. 若有 RAG 结果，用来补充背景、解释和原因
    3. 不要编造未执行得到的信息
    4. 回答简洁、自然
    5. 若存在图表路径，不要在正文重复描述文件名
    6. 若系统已经生成图表，则正文中禁止输出任何 Python / matplotlib / pyecharts 代码
    7. 禁止输出 ```python 代码块
    8. 不要描述“如何画图”，只需要给出数据概览、结论和分析
    9. 若已有图表，正文只保留：
       - 简要数据概览
       - 趋势/对比分析
       - 结论
    10. 最终答案只输出纯文本，不要使用 markdown 格式
    """.strip()

    response = llm.invoke([{"role": "user", "content": prompt}])
    response_text = clean_answer_text(response.content)

    output = {
        "messages": [AIMessage(content=response.content)],
#         "final_answer": response.content,
        "final_answer": response_text,
        "image_path": images,
        "references": all_references,
    }

    # 如果当前任务是 synthesis，则同步写入 task_results
    if current_task_id:
        try:
            task = get_current_task(state)
            if task["type"] == "synthesis":
                output["task_plan"] = update_task_status(state["task_plan"], task["task_id"], "done")
                output["task_results"] = {
                    task["task_id"]: {
                        "type": "synthesis",
                        "goal": task["goal"],
#                         "answer": response.content,
                        "answer": response_text,
                        "image_path": images,
                        "references": all_references,
                        "status": "done",
                    }
                }
                output["execution_trace"] = [{
                    "task_id": task["task_id"],
                    "type": "synthesis",
                    "goal": task["goal"],
                    "status": "done"
                }]
        except Exception:
            pass

    return output

In [ ]:
def update_task_status(task_plan: List[dict], task_id: str, new_status: str):
    updated = []
    found = False

    for task in task_plan:
        t = dict(task)
        if t["task_id"] == task_id:
            t["status"] = new_status
            found = True
        updated.append(t)

    if not found:
        raise ValueError(f"update_task_status 找不到 task_id={task_id}")

    return updated

def all_deps_done(task: dict, task_plan: List[dict]):
    deps = task.get("depends_on", [])
    if not deps:
        return True

    task_map = {t["task_id"]: t for t in task_plan}

    for dep_id in deps:
        dep_task = task_map.get(dep_id)
        if not dep_task:
            return False
        if dep_task.get("status") != "done":
            return False

    return True


def has_unfinished_tasks(task_plan: List[dict]):
    return any(t["status"] not in ("done", "failed") for t in task_plan)

In [ ]:
def scheduler(state: ParentState):
    task_plan = state.get("task_plan", [])
    task_results = state.get("task_results", {})

    updated_plan = []
    ready_tasks = []

    for task in task_plan:
        t = dict(task)

        if t["status"] in ("done", "failed"):
            updated_plan.append(t)
            continue

        if all_deps_done(t, updated_plan):
            if t["status"] == "pending":
                t["status"] = "ready"

        if t["status"] == "ready":
            ready_tasks.append(t)

        updated_plan.append(t)

    # 所有任务都完成
    if not has_unfinished_tasks(updated_plan):
        return {
            "task_plan": updated_plan,
            "current_task_id": "",
            "schedule_status": "all_done",
        }

    # 还有未完成任务，但没有 ready task，说明阻塞了
    if not ready_tasks:
        return {
            "task_plan": updated_plan,
            "current_task_id": "",
            "schedule_status": "blocked",
        }

    ready_tasks.sort(key=lambda x: (x["priority"], x["task_id"]))
    current = ready_tasks[0]

    updated_plan = update_task_status(updated_plan, current["task_id"], "running")

    return {
        "task_plan": updated_plan,
        "current_task_id": current["task_id"],
        "schedule_status": "has_task",
    }

def scheduler_route(state: ParentState) -> Literal["sql_node", "rag_node", "clarify", "merge_generate"]:
    schedule_status = state.get("schedule_status")

    if schedule_status is None:
        raise ValueError("scheduler_route 被调用时，state 中还没有 schedule_status。请检查图的边是否先经过 scheduler。")

    if schedule_status == "all_done":
        return "merge_generate"

    if schedule_status == "blocked":
        raise ValueError("存在未完成任务，但当前没有可执行任务。请检查 depends_on、task_results 或任务状态更新逻辑。")

    task = get_current_task(state)
    t = task["type"]

    if t == "sql":
        return "sql_node"
    if t == "rag":
        return "rag_node"
    if t == "clarify":
        return "clarify"
    if t == "synthesis":
        return "merge_generate"

    raise ValueError(f"未知任务类型: {t}")

In [ ]:
import os
import re
import uuid
import pandas as pd
import matplotlib.pyplot as plt
from langgraph.checkpoint.memory import MemorySaver

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

os.makedirs("./result", exist_ok=True)

PLOT_KEYWORDS = [
    "趋势", "走势", "变化", "可视化", "绘图", "画图",
    "折线图", "柱状图", "历年", "近年", "近三年", "近五年"
]

def should_plot(question: str) -> bool:
    return any(k in question for k in PLOT_KEYWORDS)

def extract_last_sql_query(messages):
    for msg in reversed(messages):
        tool_calls = getattr(msg, "tool_calls", None) or []
        for tc in tool_calls:
            if tc.get("name") == "sql_db_query":
                args = tc.get("args", {}) or {}
                sql_text = args if isinstance(args, str) else (
                    args.get("query") or
                    args.get("__arg1") or
                    args.get("input") or
                    args.get("sql")
                )
                if sql_text:
                    return sql_text
    return None

def period_sort_key(val):
    s = str(val).strip()
    m = re.match(r"(\d{4})(?:Q([1-4])|FY)", s)
    if not m:
        return (9999, 99, s)

    year = int(m.group(1))
    order = 5 if "FY" in s else int(m.group(2))
    return (year, order, s)

def pick_xy_columns(df):
    """
    自动找 x 轴和 y 轴：
    优先找 x_axis / y_axis，
    否则优先 report_period + 第一个可转成数值的列
    """
    x_candidates = ["x_axis", "report_period", "period", "日期", "年份", "year"]
    x_col = next((c for c in x_candidates if c in df.columns), df.columns[0])

    numeric_cols = []
    for c in df.columns:
        if c == x_col:
            continue
        series = pd.to_numeric(df[c], errors="coerce")
        if series.notna().any():
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"没有找到可绘图的数值列，当前列为: {list(df.columns)}")

    y_col = "y_axis" if "y_axis" in numeric_cols else numeric_cols[0]
    return x_col, y_col

def save_line_chart(df, question, out_dir="./result", prefix="chart"):
    if df is None or df.empty or len(df.columns) < 2:
        return []

    x_col, y_col = pick_xy_columns(df)

    plot_df = df[[x_col, y_col]].copy()
    plot_df[y_col] = pd.to_numeric(plot_df[y_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[y_col])

    if plot_df.empty or len(plot_df) < 2:
        return []

    if x_col in ["x_axis", "report_period", "period"]:
        plot_df["_sort_key"] = plot_df[x_col].map(period_sort_key)
        plot_df = plot_df.sort_values("_sort_key").drop(columns=["_sort_key"])

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(plot_df[x_col].astype(str), plot_df[y_col], marker="o")
    ax.set_title(question)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    plt.xticks(rotation=45)
    plt.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    file_path = os.path.join(out_dir, f"{prefix}_{uuid.uuid4().hex[:8]}.png")
    plt.savefig(file_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return [file_path]

In [ ]:
def sql_node(state: ParentState):
    task = get_current_task(state)
    goal = task["goal"]

    result = sql_graph.invoke({
        "sql_messages": [{"role": "user", "content": goal}],
        "real_query": goal,
    })

    sql_messages = result["sql_messages"]
    last_msg = sql_messages[-1]
    answer = getattr(last_msg, "content", str(last_msg))

    sql_text = extract_last_sql_query(sql_messages)
    image_paths = []

    if sql_text and should_plot(goal):
        try:
            df = pd.read_sql_query(sql_text, engine)
            image_paths = save_line_chart(
                df,
                goal,
                out_dir="./result",
                prefix="sql_plot"
            )
        except Exception as e:
            print(f"[plot error] {e}")

    return {
    "messages": [last_msg],
    "last_sql_query": sql_text,
    "image_path": image_paths,
    "task_results": {
        task["task_id"]: {
            "type": "sql",
            "goal": goal,
            "query": sql_text,
            "answer": answer,
            "image_path": image_paths,
            "status": "done",
        }
    },
    "execution_trace": [{
        "task_id": task["task_id"],
        "type": "sql",
        "goal": goal,
        "status": "done"
    }]
}

In [ ]:
def mark_current_task_done(state: ParentState):
    task_id = state["current_task_id"]
    new_plan = update_task_status(state["task_plan"], task_id, "done")

    return {
        "task_plan": new_plan
    }

In [ ]:
workflow = StateGraph(ParentState)

workflow.add_node("sql_node", sql_node) 
workflow.add_node("rag_node", rag_node) 
workflow.add_node("plan_node", plan_node) 
workflow.add_node("clarify", clarify)
workflow.add_node("merge_generate", merge_generate)
workflow.add_node("real_query", merge_query)
workflow.add_node("summarize", summarize)
workflow.add_node("scheduler", scheduler)
workflow.add_node("mark_current_task_done", mark_current_task_done)

workflow.add_edge(START, "real_query")

workflow.add_edge("real_query", "plan_node")
workflow.add_edge("plan_node", "scheduler")

workflow.add_conditional_edges(
    "scheduler",
    scheduler_route,
    {
        "sql_node": "sql_node",
        "rag_node": "rag_node",
        "clarify": "clarify",
        "merge_generate": "merge_generate",
    },
)

workflow.add_edge("sql_node", "mark_current_task_done")
workflow.add_edge("rag_node", "mark_current_task_done")
workflow.add_edge("mark_current_task_done", "scheduler")

workflow.add_edge("clarify", "summarize")
workflow.add_edge("merge_generate", "summarize")

workflow.add_edge("summarize", END)

checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from pprint import pprint

def format_msg(msg):
    if isinstance(msg, dict):
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        return f"{role}: {content}"

    msg_type = getattr(msg, "type", msg.__class__.__name__)
    content = getattr(msg, "content", str(msg))

    if msg_type == "human":
        role = "user"
    elif msg_type == "ai":
        role = "assistant"
    else:
        role = msg_type

    return f"{role}: {content}"


def run_test_case(user_query: str, thread_id: str = "test1"):
    inputs = {
        "messages": [
            {
                "role": "user",
                "content": user_query,
            }
        ],
        "query": user_query,
    }

    config = {"configurable": {"thread_id": thread_id}}

    print("=" * 100)
    print(f"用户问题: {user_query}")
    print(f"thread_id: {thread_id}")
    print("=" * 100)

    for step in graph.stream(inputs, config):
        for node_name, node_output in step.items():
            print(f"\nNode: {node_name}")

            if not isinstance(node_output, dict):
                print(node_output)
                continue

            # 1. messages
            if "messages" in node_output and node_output["messages"]:
                print("messages:")
                print(format_msg(node_output["messages"][-1]))

            # 2. real_query
            if "real_query" in node_output:
                print("real_query:")
                print(node_output["real_query"])

            # 3. route / missing_slots
            if "route" in node_output:
                print("route:")
                print(node_output["route"])

            if "missing_slots" in node_output:
                print("missing_slots:")
                pprint(node_output["missing_slots"])

            # 4. task_plan
            if "task_plan" in node_output:
                print("task_plan:")
                pprint(node_output["task_plan"])

            # 5. current_task_id
            if "current_task_id" in node_output:
                print("current_task_id:")
                print(node_output["current_task_id"])

            # 6. task_results
            if "task_results" in node_output:
                print("task_results:")
                pprint(node_output["task_results"])

            # 7. execution_trace
            if "execution_trace" in node_output:
                print("execution_trace:")
                pprint(node_output["execution_trace"])

            # 8. final_answer
            if "final_answer" in node_output:
                print("final_answer:")
                print(node_output["final_answer"])

            # 9. image_path
            if "image_path" in node_output:
                print("image_path:")
                pprint(node_output["image_path"])

            # 10. summary
            if "summary" in node_output:
                print("summary:")
                print(node_output["summary"])

        print("-" * 100)

In [ ]:
import ast
import json
import uuid
import pandas as pd


def parse_question_cell(cell):
    """
    解析 Excel 中“问题”列
    支持：
    1. 普通字符串
    2. JSON 字符串
    3. Python 列表字符串
    返回：问题列表，如 ["问题1", "问题2"]
    """
    if pd.isna(cell):
        return []

    if isinstance(cell, list):
        raw = cell
    else:
        text = str(cell).strip()
        if not text:
            return []

        text = (
            text.replace("“", '"')
                .replace("”", '"')
                .replace("‘", "'")
                .replace("’", "'")
        )

        if text.startswith("[") and text.endswith("]"):
            try:
                raw = json.loads(text)
            except Exception:
                raw = ast.literal_eval(text)
        else:
            return [text]

    questions = []
    for item in raw:
        if isinstance(item, dict):
            q = item.get("Q") or item.get("q") or item.get("question")
            if q and str(q).strip():
                questions.append(str(q).strip())
        elif isinstance(item, str):
            if item.strip():
                questions.append(item.strip())

    return questions


def normalize_image_list(x):
    """
    统一把 image / image_path 转成 list[str]
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [str(i) for i in x if str(i).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            val = json.loads(s)
            if isinstance(val, list):
                return [str(i) for i in val if str(i).strip()]
            return [s]
        except Exception:
            return [s]
    return []


def detect_chart_type(question, image_list):
    """
    根据问题内容和是否有图，给出图形格式
    你现在先用最简单稳定的规则：
    - 没图：无
    - 有图且问题包含趋势/走势/变化：折线图
    - 有图且问题包含占比/构成：饼图
    - 其他有图：图表
    """
    if not image_list:
        return "无"

    q = str(question)
    if any(k in q for k in ["趋势", "走势", "变化", "历年", "近几年", "近年"]):
        return "折线图"
    if any(k in q for k in ["占比", "构成"]):
        return "饼图"
    if any(k in q for k in ["对比", "比较", "排名"]):
        return "柱状图"

    return "图表"

def run_one_turn(question, thread_id, turn_idx=1, question_count=1):
    """
    执行单轮问答，并提取当前轮需要写入 Excel 的核心信息
    """
    # 最后一问不做 summary；单问题也自然不做
    skip_summary = (turn_idx >= question_count)

    config = {"configurable": {"thread_id": thread_id}}
    inputs = {
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ],
        "query": question,
        "question_count": question_count,
        "skip_summary": skip_summary,
    }

    for _ in graph.stream(inputs, config):
        pass

    state = graph.get_state(config).values

    task_results = state.get("task_results", {})

    # 1. SQL
    sql_query = state.get("last_sql_query", "")

    if not sql_query:
        for _, result in task_results.items():
            if result.get("type") == "sql" and result.get("query"):
                sql_query = result.get("query", "")

    # 2. 最终文本回答
    answer_text = state.get("final_answer", "")

    if not answer_text:
        # 如果没有 synthesis，就从最后一个任务结果里兜底取
        for _, result in task_results.items():
            if result.get("type") in ("sql", "rag", "synthesis") and result.get("answer"):
                answer_text = result.get("answer", "")
                
    answer_text = clean_answer_text(answer_text)

    # 3. 图表
    image_list = normalize_image_list(state.get("image_path"))

    if not image_list:
        for _, result in task_results.items():
            if result.get("type") == "sql" and result.get("image_path"):
                image_list = normalize_image_list(result.get("image_path"))
                break

    # 4. references
    references = state.get("references", [])
    if not references:
        for _, result in task_results.items():
            if result.get("type") == "rag" and result.get("references"):
                references.extend(result.get("references", []))

    answer_obj = {
        "content": answer_text
    }

    if image_list:
        answer_obj["image"] = image_list

    if references:
        answer_obj["references"] = references

    return {
        "Q": question,
        "A": answer_obj,
        "sql_query": sql_query,
        "image_list": image_list,
    }

def run_excel_batch(
    xlsx_path,
    sheet_name=0,
    id_col="编号",
    question_col="问题",
    output_path="./result/批量问答结果.xlsx"
):
    """
    输出格式与截图一致：
    编号 | 问题 | SQL 查询语句 | 图形格式 | 回答
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    df.columns = [str(c).strip() for c in df.columns]

    if id_col not in df.columns:
        raise ValueError(f"Excel 中未找到列：{id_col}，当前列名：{df.columns.tolist()}")
    if question_col not in df.columns:
        raise ValueError(f"Excel 中未找到列：{question_col}，当前列名：{df.columns.tolist()}")

    batch_tag = uuid.uuid4().hex[:8]
    result_rows = []

    for _, row in df.iterrows():
        row_id = str(row[id_col]).strip()
        question_list = parse_question_cell(row[question_col])

        if not question_list:
            result_rows.append({
                "编号": row_id,
                "问题": "",
                "SQL 查询语句": "",
#                 "图形格式": "无",
                "回答": "",
            })
            continue

        thread_id = f"excel_{batch_tag}_{row_id}"

        qa_list = []
        last_sql = ""
        final_images = []
        image_counter = 0

        question_count = len(question_list)

        for turn_idx, q in enumerate(question_list, start=1):
            try:
                turn_result = run_one_turn(
                    q,
                    thread_id,
                    turn_idx=turn_idx,
                    question_count=question_count
                )
                
                # ===== 新增：把当前轮生成的图片改名 =====
                renamed_images = []
                old_images = turn_result.get("image_list", [])

                for old_path in old_images:
                    if not old_path:
                        continue
                    if not os.path.exists(old_path):
                        renamed_images.append(old_path)
                        continue

                    image_counter += 1

                    ext = os.path.splitext(old_path)[1] or ".png"
                    new_path = os.path.join(
                        os.path.dirname(old_path),
                        f"{row_id}_{image_counter}{ext}"
                    )

                    # 如果目标文件已存在，先删掉，避免重命名失败
                    if os.path.exists(new_path):
                        os.remove(new_path)

                    os.replace(old_path, new_path)
                    renamed_images.append(new_path)

                # 回写 turn_result，保证后面写入 Excel 的路径也是新路径
                if renamed_images:
                    turn_result["image_list"] = renamed_images
                    if "A" in turn_result and isinstance(turn_result["A"], dict):
                        turn_result["A"]["image"] = renamed_images
        # ===== 新增结束 =====

                qa_list.append({
                    "Q": turn_result["Q"],
                    "A": turn_result["A"]
                })

                if turn_result["sql_query"]:
                    last_sql = turn_result["sql_query"]

                if turn_result["image_list"]:
                    final_images = turn_result["image_list"]

            except Exception as e:
                qa_list.append({
                    "Q": q,
                    "A": {
                        "content": f"系统异常: {str(e)}"
                    }
                })

        chart_type = detect_chart_type(
            question=" ".join(question_list),
            image_list=final_images
        )

        result_rows.append({
            "编号": row_id,
            "问题": json.dumps([{"Q": q} for q in question_list], ensure_ascii=False),
            "SQL 查询语句": last_sql if last_sql else "无",
#             "图形格式": chart_type,
            "回答": json.dumps(qa_list, ensure_ascii=False, indent=2),
        })

    result_df = pd.DataFrame(result_rows)
    result_df.to_excel(output_path, index=False)
    print(f"批量问答完成，结果保存到：{output_path}")
    return result_df

In [ ]:
import time

start = time.time()

run_test_case("2024年利润最高的top10企业是哪些？这些企业的利润、销售额年同比是多少？年同比上涨幅度最大的是哪家企业？", thread_id="test111")

end = time.time()
print("运行时间：", end - start, "秒")

In [ ]:
import time

start = time.time()

result_df = run_excel_batch(
    xlsx_path=r"示例数据\附件6：问题汇总.xlsx",
    sheet_name=0,
    id_col="编号",
    question_col="问题",  
    output_path=r"示例数据\任务三批量问答结果.xlsx"
)

end = time.time()
print("运行时间：", end - start, "秒")

In [ ]:
import time

start = time.time()

run_test_case("华润三九2023第一季度的每股收益是多少", thread_id="test102")

end = time.time()
print("运行时间：", end - start, "秒")

In [ ]:
import time

start = time.time()

run_test_case("华润三九每股收益是多少", thread_id="test103")

end = time.time()
print("运行时间：", end - start, "秒")

run_test_case("2023第一季度的", thread_id="test103")

end1 = time.time()
print("运行时间：", end1 - end, "秒")

In [ ]:
import time

start = time.time()

run_test_case("国家医保目录新增的中药产品有哪些", thread_id="test104")
run_test_case("你的依据是什么", thread_id="test104")

end = time.time()
print("运行时间：", end - start, "秒")

In [ ]:
import time

start = time.time()

run_test_case("华润三九近几年的每股收益的变化趋势", thread_id="test105")

end = time.time()
print("运行时间：", end - start, "秒")